# Final 5-Class FinBERT Fine-Tuning Notebook — Kaggle

This notebook fine-tunes the **financial event classifier** for the hackathon pipeline.

## Fixed task

The model predicts exactly five event classes:

| ID | Label |
|---:|---|
| 0 | Geopolitical |
| 1 | Macroeconomic |
| 2 | Credit Event |
| 3 | Merger / Acquisition |
| 4 | Product Launch |

## Model

We start from:

`nickmuchi/finbert-tone-finetuned-finance-topic-classification`

That checkpoint was already fine-tuned for financial-topic classification with a 20-class head. We keep the pretrained financial encoder weights and replace the old classification head with a new **5-class head**.

## Data

Attach the **complete output of the source-aware preprocessing notebook** as a Kaggle Input. This notebook automatically searches `/kaggle/input/` for:

- `event_train.parquet` (CSV fallback)
- `event_validation.parquet` (CSV fallback)
- `event_test.parquet` (CSV fallback)

The current source-aware preprocessing produced a strongly imbalanced real dataset. We therefore use an **imbalance-aware training sampler** and ordinary cross-entropy instead of also applying the extreme class weights. Validation and test remain untouched real data.

## Evaluation

Primary metric: **Macro-F1**

Also reported:

- accuracy
- weighted F1
- macro precision / recall
- per-class precision / recall / F1 / support
- confusion matrix
- normalized confusion matrix
- test predictions with probabilities

## Long documents

The baseline uses **head truncation to 512 tokens**. This keeps the architecture simple and fast for the hackathon. Chunking can be added later if needed.

## Important Kaggle setup

For practical training speed, enable a GPU:

**Notebook settings → Accelerator → GPU**

CPU fallback exists, but GPU is strongly recommended.

## 1. Install the required libraries

Install Transformers, Datasets, Accelerate, scikit-learn, and Parquet support.

In [ ]:
%pip -q install -U "transformers>=4.46,<5" "datasets>=3,<4" "accelerate>=1,<2" "scikit-learn>=1.4" pyarrow safetensors

## 2. Imports, reproducibility, and hardware detection

Set a fixed random seed and detect whether Kaggle provided CUDA. Mixed precision is enabled automatically when supported.

In [ ]:
import os
import gc
import json
import math
import random
import shutil
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Device:", DEVICE)
print("Hardware:", GPU_NAME)
print("PyTorch:", torch.__version__)

In [ ]:
# Hugging Face login from Kaggle Secret

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 3. Fixed experiment configuration

All important choices are in one place. The defaults are selected for a strong baseline while keeping training practical on Kaggle.

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

MODEL_NAME = "nickmuchi/finbert-tone-finetuned-finance-topic-classification"

LABEL2ID = {
    "Geopolitical": 0,
    "Macroeconomic": 1,
    "Credit Event": 2,
    "Merger / Acquisition": 3,
    "Product Launch": 4,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

MAX_LENGTH = 512

# Training settings
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

# Imbalance handling:
# 1.0 = full inverse-frequency sampling (very aggressive)
# 0.5 = inverse-sqrt sampling (recommended baseline)
# 0.0 = natural distribution
SAMPLER_POWER = 0.50

# Early stopping protects against overfitting, especially Credit Event.
EARLY_STOPPING_PATIENCE = 2

# Trainer will pick the best checkpoint by Macro-F1.
BEST_MODEL_METRIC = "macro_f1"

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()

MODEL_OUT = WORKDIR / "finbert_event_classifier_5class"
CHECKPOINT_OUT = WORKDIR / "finbert_training_checkpoints"

print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Epochs:", NUM_EPOCHS)
print("Sampler power:", SAMPLER_POWER)

## 4. Automatically locate preprocessing outputs

The notebook recursively searches all attached Kaggle inputs. Parquet is preferred; CSV is accepted as a fallback.

In [ ]:
def find_file(root: Path, exact_names):
    hits = []
    if root.exists():
        for name in exact_names:
            hits.extend(root.rglob(name))
    hits = sorted(set(hits), key=lambda p: (p.suffix.lower() != ".parquet", str(p)))
    return hits

def resolve_split(stem):
    hits = find_file(INPUT_ROOT, [f"{stem}.parquet", f"{stem}.csv"])
    print(f"\nCandidates for {stem}:")
    for h in hits:
        print(" ", h)
    if not hits:
        raise FileNotFoundError(
            f"Could not find {stem}.parquet or {stem}.csv under {INPUT_ROOT}. "
            "Attach the complete output of the preprocessing notebook as a Kaggle Input."
        )
    chosen = hits[0]
    print("Selected:", chosen)
    return chosen

TRAIN_PATH = resolve_split("event_train")
VAL_PATH = resolve_split("event_validation")
TEST_PATH = resolve_split("event_test")

def load_table(path):
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)

train_df = load_table(TRAIN_PATH)
val_df = load_table(VAL_PATH)
test_df = load_table(TEST_PATH)

print("\nShapes")
print("train     :", train_df.shape)
print("validation:", val_df.shape)
print("test      :", test_df.shape)

## 5. Validate the dataset before training

This cell verifies label IDs, missing text, duplicate leakage, and class counts. Training stops immediately if the core schema is inconsistent.

In [ ]:
REQUIRED = {"text", "label", "label_id"}

def validate_split(df, split_name):
    missing = REQUIRED - set(df.columns)
    if missing:
        raise ValueError(f"{split_name} missing required columns: {sorted(missing)}")

    if df["text"].isna().any():
        raise ValueError(f"{split_name} contains null text.")

    empty = df["text"].astype(str).str.strip().eq("")
    if empty.any():
        raise ValueError(f"{split_name} contains {int(empty.sum())} empty texts.")

    bad_labels = set(df["label"].astype(str).unique()) - set(LABEL2ID)
    if bad_labels:
        raise ValueError(f"{split_name} contains unknown labels: {bad_labels}")

    pairs = df[["label", "label_id"]].drop_duplicates()
    for r in pairs.itertuples(index=False):
        expected = LABEL2ID[str(r.label)]
        if int(r.label_id) != expected:
            raise ValueError(
                f"{split_name}: label mapping mismatch for {r.label}: "
                f"got {r.label_id}, expected {expected}"
            )

validate_split(train_df, "train")
validate_split(val_df, "validation")
validate_split(test_df, "test")

def norm_text(x):
    return " ".join(str(x).lower().split())

train_norm = set(train_df["text"].astype(str).map(norm_text))
val_norm = set(val_df["text"].astype(str).map(norm_text))
test_norm = set(test_df["text"].astype(str).map(norm_text))

assert train_norm.isdisjoint(val_norm), "Exact text leakage between train and validation."
assert train_norm.isdisjoint(test_norm), "Exact text leakage between train and test."
assert val_norm.isdisjoint(test_norm), "Exact text leakage between validation and test."

print("Class counts:")
counts_table = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).reindex(LABEL2ID.keys()).fillna(0).astype(int)

display(counts_table)
print("\nDataset validation: PASSED")

## 6. Visualize the real training imbalance

This is only an audit plot. We do not alter validation/test distributions.

In [ ]:
ax = train_df["label"].value_counts().reindex(LABEL2ID.keys(), fill_value=0).plot(
    kind="bar", figsize=(9, 4)
)
ax.set_title("REAL training class distribution")
ax.set_xlabel("")
ax.set_ylabel("Rows")
plt.tight_layout()
plt.savefig(WORKDIR / "plot_train_class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Load the financial tokenizer and replace the old 20-class head

`ignore_mismatched_sizes=True` keeps the pretrained financial encoder while creating a fresh five-output classification layer.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    token=HF_TOKEN
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
    token=HF_TOKEN
)

model.config.problem_type = "single_label_classification"

print("Model architecture loaded.")
print("num_labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

## 8. Convert Pandas splits to Hugging Face Datasets

Only the text and integer target label are passed into the model. Original metadata remain in the source Parquet files.

In [ ]:
def to_hf_dataset(df):
    tmp = df[["text", "label_id"]].copy()
    tmp = tmp.rename(columns={"label_id": "labels"})
    tmp["labels"] = tmp["labels"].astype(int)
    return Dataset.from_pandas(tmp, preserve_index=False)

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

print(train_ds)
print(val_ds)
print(test_ds)

## 9. Tokenize with 512-token head truncation

Padding is dynamic at batch time. Long documents are truncated from the end so the first 512 tokens are used.

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

train_tok = train_ds.map(tokenize_batch, batched=True, desc="Tokenizing train")
val_tok = val_ds.map(tokenize_batch, batched=True, desc="Tokenizing validation")
test_tok = test_ds.map(tokenize_batch, batched=True, desc="Tokenizing test")

# Remove raw text from tensors sent to the model.
train_tok = train_tok.remove_columns(["text"])
val_tok = val_tok.remove_columns(["text"])
test_tok = test_tok.remove_columns(["text"])

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest",
    return_tensors="pt",
)

print(train_tok)

## 10. Build the imbalance-aware sampler

We do **not** use the old extreme class weights in the loss. Instead, examples from minority classes receive a higher probability of being sampled. `SAMPLER_POWER=0.5` uses inverse-square-root frequency, which is less aggressive than forcing every class to appear equally often.

In [ ]:
train_labels = train_df["label_id"].astype(int).to_numpy()
class_counts = np.bincount(train_labels, minlength=5)

class_sampling_weights = np.power(
    1.0 / np.maximum(class_counts, 1),
    SAMPLER_POWER
)

sample_weights = class_sampling_weights[train_labels]

sampler_audit = pd.DataFrame({
    "label_id": range(5),
    "label": [ID2LABEL[i] for i in range(5)],
    "train_count": class_counts,
    "sampling_weight": class_sampling_weights,
})

display(sampler_audit)

print(
    "\nImportant: ordinary cross-entropy will be used. "
    "We are NOT combining this sampler with the old 27x Credit loss weight."
)

## 11. Define metrics

Macro-F1 treats every class equally and is therefore the main selection metric. Per-class metrics are calculated separately after training.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, preds)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "macro_precision": float(p_macro),
        "macro_recall": float(r_macro),
        "macro_f1": float(f1_macro),
        "weighted_f1": float(f1_weighted),
    }

## 12. Custom Trainer with the sampler

Hugging Face Trainer normally shuffles the real distribution. This subclass replaces only the **training** DataLoader with our WeightedRandomSampler. Validation and test remain natural real distributions.

In [ ]:
class ImbalanceAwareTrainer(Trainer):
    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires train_dataset.")

        generator = torch.Generator()
        generator.manual_seed(SEED)

        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
            generator=generator,
        )

        return DataLoader(
            self.train_dataset,
            batch_size=self._train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            drop_last=self.args.dataloader_drop_last,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

## 13. Training arguments

We evaluate and save once per epoch, use early stopping, and restore the checkpoint with the best validation Macro-F1.

In [ ]:
use_fp16 = bool(torch.cuda.is_available())
use_bf16 = False

# Prefer BF16 on newer GPUs when supported.
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    use_bf16 = True
    use_fp16 = False

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_OUT),

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model=BEST_MODEL_METRIC,
    greater_is_better=True,
    save_total_limit=2,

    fp16=use_fp16,
    bf16=use_bf16,

    dataloader_num_workers=2,
    dataloader_pin_memory=torch.cuda.is_available(),

    report_to="none",
    seed=SEED,
    data_seed=SEED,

    remove_unused_columns=True,
)

print("fp16:", use_fp16)
print("bf16:", use_bf16)
print(training_args)

## 14. Create the Trainer

Early stopping watches validation Macro-F1. The test split is not used during optimization.

In [ ]:
trainer = ImbalanceAwareTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

print("Trainer ready.")

## 15. Fine-tune FinBERT

This is the main training cell. Runtime depends mostly on the Kaggle GPU. Checkpoints are written to `/kaggle/working/finbert_training_checkpoints`.

In [ ]:
train_result = trainer.train()

print("\nTraining complete.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation metric:", trainer.state.best_metric)

train_metrics = train_result.metrics
train_metrics

## 16. Evaluate the best model on validation

This gives the metrics used for model selection and a detailed per-class report.

In [ ]:
val_metrics = trainer.evaluate(val_tok, metric_key_prefix="validation")
print(json.dumps(val_metrics, indent=2))

val_pred_output = trainer.predict(val_tok, metric_key_prefix="validation_pred")
val_preds = np.argmax(val_pred_output.predictions, axis=-1)
val_true = val_pred_output.label_ids

print("\nVALIDATION CLASSIFICATION REPORT")
print(
    classification_report(
        val_true,
        val_preds,
        labels=list(range(5)),
        target_names=[ID2LABEL[i] for i in range(5)],
        digits=4,
        zero_division=0,
    )
)

## 17. Final evaluation on the frozen real test split

The test split is touched only after training is complete. Because Credit Event currently has very few real test examples, its individual metric should be interpreted cautiously; Macro-F1 is still reported for the complete five-class task.

In [ ]:
test_output = trainer.predict(test_tok, metric_key_prefix="test")

test_logits = test_output.predictions
test_true = test_output.label_ids
test_preds = np.argmax(test_logits, axis=-1)

test_metrics = compute_metrics((test_logits, test_true))

print("TEST SUMMARY")
print(json.dumps(test_metrics, indent=2))

report_dict = classification_report(
    test_true,
    test_preds,
    labels=list(range(5)),
    target_names=[ID2LABEL[i] for i in range(5)],
    output_dict=True,
    digits=6,
    zero_division=0,
)

report_text = classification_report(
    test_true,
    test_preds,
    labels=list(range(5)),
    target_names=[ID2LABEL[i] for i in range(5)],
    digits=4,
    zero_division=0,
)

print("\nTEST CLASSIFICATION REPORT")
print(report_text)

## 18. Confusion matrices

Both raw-count and row-normalized matrices are saved for the hackathon report/dashboard analysis.

In [ ]:
labels_order = list(range(5))
display_labels = [ID2LABEL[i] for i in labels_order]

cm = confusion_matrix(test_true, test_preds, labels=labels_order)
cm_norm = confusion_matrix(test_true, test_preds, labels=labels_order, normalize="true")

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
disp.plot(ax=ax, xticks_rotation=35, values_format="d")
ax.set_title("Test Confusion Matrix — Counts")
plt.tight_layout()
plt.savefig(WORKDIR / "confusion_matrix_test_counts.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=display_labels)
disp.plot(ax=ax, xticks_rotation=35, values_format=".2f")
ax.set_title("Test Confusion Matrix — Row Normalized")
plt.tight_layout()
plt.savefig(WORKDIR / "confusion_matrix_test_normalized.png", dpi=180, bbox_inches="tight")
plt.show()

## 19. Save per-class test metrics

This creates a compact CSV that can be used directly in the project report or presentation.

In [ ]:
per_class_rows = []

for label_name in LABEL2ID:
    r = report_dict[label_name]
    per_class_rows.append({
        "label": label_name,
        "label_id": LABEL2ID[label_name],
        "precision": r["precision"],
        "recall": r["recall"],
        "f1": r["f1-score"],
        "support": int(r["support"]),
    })

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)

per_class_df.to_csv(WORKDIR / "test_per_class_metrics.csv", index=False)

## 20. Save every test prediction and class probability

These outputs make error analysis and downstream integration easier.

In [ ]:
# Stable softmax probabilities.
probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()

predictions_df = test_df.copy().reset_index(drop=True)
predictions_df["true_label_id"] = test_true
predictions_df["pred_label_id"] = test_preds
predictions_df["pred_label"] = [ID2LABEL[int(i)] for i in test_preds]
predictions_df["correct"] = predictions_df["true_label_id"] == predictions_df["pred_label_id"]

for i in range(5):
    safe_name = ID2LABEL[i].lower().replace(" ", "_").replace("/", "or")
    predictions_df[f"prob_{i}_{safe_name}"] = probs[:, i]

predictions_df.to_csv(WORKDIR / "test_predictions_with_probabilities.csv", index=False)
predictions_df.to_parquet(WORKDIR / "test_predictions_with_probabilities.parquet", index=False)

display(predictions_df.head())
print("Test accuracy from saved predictions:", predictions_df["correct"].mean())

## 21. Save the final model and tokenizer

This directory is what the later API/risk-engine notebook should load for event classification.

In [ ]:
MODEL_OUT.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(MODEL_OUT))
tokenizer.save_pretrained(str(MODEL_OUT))

print("Saved model to:", MODEL_OUT)
print("Files:")
for p in sorted(MODEL_OUT.iterdir()):
    print(" ", p.name)

## 22. Save a complete training/evaluation report

This JSON records the model, dataset sizes, hyperparameters, validation/test metrics, and known limitations.

In [ ]:
final_report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": MODEL_NAME,
    "task": "5-class financial event classification",
    "label_mapping": LABEL2ID,
    "dataset": {
        "train_rows": int(len(train_df)),
        "validation_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_class_counts": {
            str(k): int(v) for k, v in train_df["label"].value_counts().to_dict().items()
        },
        "validation_class_counts": {
            str(k): int(v) for k, v in val_df["label"].value_counts().to_dict().items()
        },
        "test_class_counts": {
            str(k): int(v) for k, v in test_df["label"].value_counts().to_dict().items()
        },
    },
    "training": {
        "max_length": MAX_LENGTH,
        "num_epochs_requested": NUM_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "sampler": "WeightedRandomSampler",
        "sampler_power": SAMPLER_POWER,
        "loss": "ordinary cross-entropy from AutoModelForSequenceClassification",
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "best_model_metric": BEST_MODEL_METRIC,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_metric": trainer.state.best_metric,
        "hardware": GPU_NAME,
    },
    "validation_metrics": {
        str(k): (float(v) if isinstance(v, (int, float, np.floating)) else v)
        for k, v in val_metrics.items()
    },
    "test_summary_metrics": {
        str(k): float(v) for k, v in test_metrics.items()
    },
    "test_classification_report": report_dict,
    "known_limitations": [
        "Credit Event has few real validation/test examples; its per-class metric is statistically unstable.",
        "Geopolitical is dominated by GDELT-style text, creating possible source-style shortcut learning.",
        "Macroeconomic is dominated by Twitter Financial News Topic text.",
        "Long documents are head-truncated to 512 tokens in this baseline.",
        "Synthetic augmentation is intentionally deferred until after the full architecture works."
    ]
}

with open(WORKDIR / "finbert_event_classifier_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2)

with open(WORKDIR / "test_classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

print(json.dumps(final_report["test_summary_metrics"], indent=2))

## 23. Quick inference helper

Use this function to test arbitrary financial text. The same logic can later be copied into the unified Risk Engine API.

In [ ]:
# Reloading is unnecessary here because Trainer already holds the best model.
inference_model = trainer.model
inference_model.eval()

def predict_event(text):
    encoded = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    )
    encoded = {k: v.to(inference_model.device) for k, v in encoded.items()}

    with torch.no_grad():
        logits = inference_model(**encoded).logits
        p = torch.softmax(logits, dim=-1)[0].detach().cpu().numpy()

    pred_id = int(np.argmax(p))

    return {
        "label_id": pred_id,
        "label": ID2LABEL[pred_id],
        "confidence": float(p[pred_id]),
        "class_probabilities": {
            ID2LABEL[i]: float(p[i]) for i in range(5)
        },
    }

# Example — replace with any financial event text.
example = "The company filed for Chapter 11 protection after missing a scheduled bond payment."
print(json.dumps(predict_event(example), indent=2))

## 24. Package the trained model for easy download

Creates a ZIP containing the model/tokenizer configuration. Kaggle will also preserve the unzipped directory in notebook outputs.

In [ ]:
archive_base = WORKDIR / "finbert_event_classifier_5class"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(MODEL_OUT)
)

print("Model archive:", archive_path)

## 25. Final output checklist

After a successful run, the important outputs in `/kaggle/working/` are:

### Model
- `finbert_event_classifier_5class/`
- `finbert_event_classifier_5class.zip`

### Metrics / evaluation
- `finbert_event_classifier_report.json`
- `test_per_class_metrics.csv`
- `test_classification_report.txt`
- `confusion_matrix_test_counts.png`
- `confusion_matrix_test_normalized.png`

### Predictions
- `test_predictions_with_probabilities.csv`
- `test_predictions_with_probabilities.parquet`

### Training audit
- `plot_train_class_distribution.png`
- `finbert_training_checkpoints/`

Once this notebook finishes, **do not immediately optimize it further**. First inspect the validation/test Macro-F1 and per-class results. Then the event classifier can be connected to the rest of the architecture:

**text → event classifier → sentiment model → impact engine → structured risk signal → downstream stress-test/dashboard**

Synthetic augmentation and more advanced long-document chunking can be added later if needed.